In [1]:
"""Complete training pipeline for defect detection models."""

import os
import sys
from pathlib import Path

# Fix the import paths
from abbvisionsystem.training_pipeline.data_manager import organize_dataset, prepare_yolo_dataset, generate_synthetic_defects
from abbvisionsystem.training_pipeline.yolo_trainer import YOLODefectDetector, create_multi_object_test_images
from abbvisionsystem.training_pipeline.resnet_trainer import DefectClassificationModel


def run_complete_pipeline(
    source_data_dir: str,
    use_yolo: bool = True,
    use_classification: bool = True,
    train_yolo_epochs: int = 100,
    train_classification_epochs: int = 50
):
    """Run complete training pipeline for both YOLO and classification models."""
    
    print("🚀 Starting Complete Defect Detection Training Pipeline")
    print("=" * 60)
    
    # Step 1: Organize dataset
    print("\n📁 Step 1: Organizing dataset...")
    classification_dataset = "defect_detection_dataset"
    organize_dataset(source_data_dir, classification_dataset)
    
    # Step 2: Prepare YOLO dataset
    print("\n🎯 Step 2: Preparing YOLO dataset...")
    yolo_dataset_yaml = prepare_yolo_dataset(classification_dataset, "yolo_dataset")
    
    # Step 3: Create multi-object test images
    print("\n🖼️ Step 3: Creating multi-object test images...")
    create_multi_object_test_images(
        f"{classification_dataset}/test",
        "multi_object_test",
        images_per_composition=30
    )
    
    results = {}
    
    # Step 4: Train YOLO model (recommended for your use case)
    if use_yolo:
        print("\n🤖 Step 4: Training YOLOv8 model...")
        yolo_detector = YOLODefectDetector()
        
        try:
            best_yolo_weights = yolo_detector.train(
                dataset_yaml=yolo_dataset_yaml,
                epochs=train_yolo_epochs,
                imgsz=640,
                batch=16,
                project='trained_models',
                name='yolo_defect_detector'
            )
            
            # Evaluate YOLO
            print("\n📊 Evaluating YOLO model...")
            yolo_results = yolo_detector.evaluate_on_test_set(f"{classification_dataset}/test")
            results['yolo'] = yolo_results
            
            print(f"YOLO Results:")
            print(f"  Accuracy: {yolo_results['accuracy']:.4f}")
            print(f"  Precision: {yolo_results['precision']:.4f}")
            print(f"  Recall: {yolo_results['recall']:.4f}")
            print(f"  F1 Score: {yolo_results['f1_score']:.4f}")
            
        except Exception as e:
            print(f"YOLO training failed: {e}")
            results['yolo'] = None
    
    # Step 5: Train classification model (for comparison)
    if use_classification:
        print("\n🧠 Step 5: Training ResNet50V2 classification model...")
        classifier = DefectClassificationModel()
        classifier.build_model()
        
        try:
            # Prepare data
            train_gen, val_gen = classifier.prepare_data_generators(
                f"{classification_dataset}/train",
                f"{classification_dataset}/validation"
            )
            
            # Train
            classifier.train(
                train_gen, val_gen,
                epochs=train_classification_epochs,
                model_name="resnet_defect_classifier"
            )
            
            # Evaluate - fix the test generator creation
            test_datagen = classifier.prepare_data_generators(
                f"{classification_dataset}/test",
                f"{classification_dataset}/test"  # Using same directory
            )[1]  # Use validation generator (no augmentation)
            
            classification_results = classifier.evaluate(test_datagen)
            results['classification'] = classification_results
            
            # Save model
            classifier.save_model("resnet_defect_classifier")
            
            print(f"Classification Results:")
            print(f"  Accuracy: {classification_results['test_accuracy']:.4f}")
            print(f"  Precision: {classification_results['test_precision']:.4f}")
            print(f"  Recall: {classification_results['test_recall']:.4f}")
            
        except Exception as e:
            print(f"Classification training failed: {e}")
            results['classification'] = None
    
    # Step 6: Compare models
    print("\n📈 Step 6: Model Comparison Summary")
    print("=" * 40)
    
    if results.get('yolo') and results.get('classification'):
        print("Model Performance Comparison:")
        print(f"{'Metric':<15} {'YOLO':<10} {'ResNet50V2':<12}")
        print("-" * 37)
        print(f"{'Accuracy':<15} {results['yolo']['accuracy']:<10.4f} {results['classification']['test_accuracy']:<12.4f}")
        print(f"{'Precision':<15} {results['yolo']['precision']:<10.4f} {results['classification']['test_precision']:<12.4f}")
        print(f"{'Recall':<15} {results['yolo']['recall']:<10.4f} {results['classification']['test_recall']:<12.4f}")
        
        # Calculate F1 for classification
        precision = results['classification']['test_precision']
        recall = results['classification']['test_recall']
        f1_classification = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        print(f"{'F1 Score':<15} {results['yolo']['f1_score']:<10.4f} {f1_classification:<12.4f}")
    
    print("\n✅ Pipeline completed successfully!")
    print("\n🎯 RECOMMENDATION FOR YOUR USE CASE:")
    print("Since you need to detect multiple objects in real-world images,")
    print("YOLOv8 is the better choice as it can:")
    print("  • Detect multiple objects simultaneously")
    print("  • Provide bounding box locations")
    print("  • Handle varying numbers of objects per image")
    print("  • Scale better to production environments")
    
    return results


# Test function to check if everything is properly set up
def test_pipeline_setup():
    """Test if all components are properly set up."""
    print("🔍 Testing pipeline setup...")
    
    try:
        from abbvisionsystem.training_pipeline.data_manager import organize_dataset
        print("✅ data_manager import successful")
    except ImportError as e:
        print(f"❌ data_manager import failed: {e}")
        return False
    
    try:
        from abbvisionsystem.training_pipeline.yolo_trainer import YOLODefectDetector
        print("✅ yolo_trainer import successful")
    except ImportError as e:
        print(f"❌ yolo_trainer import failed: {e}")
        return False
    
    try:
        from abbvisionsystem.training_pipeline.resnet_trainer import DefectClassificationModel
        print("✅ resnet_trainer import successful")
    except ImportError as e:
        print(f"❌ resnet_trainer import failed: {e}")
        return False
    
    # Check if ultralytics is available for YOLO
    try:
        from ultralytics import YOLO
        print("✅ ultralytics available")
    except ImportError:
        print("⚠️  ultralytics not installed. Install with: pip install ultralytics")
    
    # Check if tensorflow is available
    try:
        import tensorflow as tf
        print(f"✅ tensorflow {tf.__version__} available")
    except ImportError:
        print("❌ tensorflow not installed")
        return False
    
    print("✅ Pipeline setup test completed successfully!")
    return True


if __name__ == "__main__":
    # First test the setup
    if not test_pipeline_setup():
        print("❌ Setup test failed. Please fix the issues above.")
        exit(1)
    
    # Run the complete pipeline
    source_dir = "data/choco-pie"  # Update this path
    
    if not os.path.exists(source_dir):
        print(f"Source directory {source_dir} not found!")
        print("Please update the source_dir variable to point to your data.")
        print("Expected structure:")
        print("data/choco-pie/")
        print("├── good/")
        print("│   ├── image1.JPG")
        print("│   └── image2.JPG")
        print("└── defect/")
        print("    ├── defect1.JPG")
        print("    └── defect2.JPG")
    else:
        # Check data structure
        good_dir = os.path.join(source_dir, "good")
        defect_dir = os.path.join(source_dir, "defect")
        
        if not os.path.exists(good_dir):
            print(f"❌ 'good' directory not found in {source_dir}")
            exit(1)
        if not os.path.exists(defect_dir):
            print(f"❌ 'defect' directory not found in {source_dir}")
            exit(1)
            
        good_files = [f for f in os.listdir(good_dir) if f.endswith(('.JPG', '.jpg', '.png', '.bmp'))]
        defect_files = [f for f in os.listdir(defect_dir) if f.endswith(('.JPG', '.jpg', '.png', '.bmp'))]
        
        print(f"📊 Dataset Summary:")
        print(f"  Normal samples: {len(good_files)}")
        print(f"  Defect samples: {len(defect_files)}")
        
        if len(good_files) == 0 or len(defect_files) == 0:
            print("❌ Insufficient data. Need at least 1 image in each category.")
            exit(1)
        
        # Run pipeline
        results = run_complete_pipeline(
            source_data_dir=source_dir,
            use_yolo=True,
            use_classification=True,
            train_yolo_epochs=50,  # Reduced for testing
            train_classification_epochs=25  # Reduced for testing
        )

🔍 Testing pipeline setup...
✅ data_manager import successful
✅ yolo_trainer import successful
✅ resnet_trainer import successful
✅ ultralytics available
✅ tensorflow 2.19.0 available
✅ Pipeline setup test completed successfully!
📊 Dataset Summary:
  Normal samples: 23
  Defect samples: 26
🚀 Starting Complete Defect Detection Training Pipeline

📁 Step 1: Organizing dataset...
Dataset organized into defect_detection_dataset

🎯 Step 2: Preparing YOLO dataset...
YOLO dataset prepared in yolo_dataset

🖼️ Step 3: Creating multi-object test images...
Created 30 multi-object test images in multi_object_test

🤖 Step 4: Training YOLOv8 model...
Model loaded from yolov8n.pt
New https://pypi.org/project/ultralytics/8.3.177 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.141 🚀 Python-3.12.10 torch-2.7.0 CPU (Apple M1 Pro)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_

train: Scanning /Users/ducle/Library/CloudStorage/OneDrive-RMITUniversity/Courses/OENG1183 - Capstone A/abb-capstone/yolo_dataset/labels/train... 34 images, 16 backgrounds, 0 corrupt: 100%|██████████| 34/34 [00:00<00:00, 1698.04it/s]

train: New cache created: /Users/ducle/Library/CloudStorage/OneDrive-RMITUniversity/Courses/OENG1183 - Capstone A/abb-capstone/yolo_dataset/labels/train.cache


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5472.8±3046.5 MB/s, size: 462.7 KB)


val: Scanning /Users/ducle/Library/CloudStorage/OneDrive-RMITUniversity/Courses/OENG1183 - Capstone A/abb-capstone/yolo_dataset/labels/val... 6 images, 3 backgrounds, 0 corrupt: 100%|██████████| 6/6 [00:00<00:00, 5235.25it/s]

...

INFO:tensorflow:Assets written to: /var/folders/8w/rnt4yvg55s123wgbg_0r__cw0000gn/T/tmpzcpjgi65/assets


INFO:tensorflow:Assets written to: /var/folders/8w/rnt4yvg55s123wgbg_0r__cw0000gn/T/tmpzcpjgi65/assets


Saved artifact at '/var/folders/8w/rnt4yvg55s123wgbg_0r__cw0000gn/T/tmpzcpjgi65'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_190')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  13469157328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13553334416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13468272400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13469156944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13469157520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13152876880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13469157136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13432419728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13432420496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13432420112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134187

W0000 00:00:1754910573.531345 1591418 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1754910573.531365 1591418 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1754910573.622443 1591418 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled


Model saved in multiple formats:
- H5: trained_models/resnet_defect_classifier.h5
- Keras: trained_models/resnet_defect_classifier.keras
- TFLite: trained_models/resnet_defect_classifier.tflite
Classification Results:
  Accuracy: 0.7778
  Precision: 1.0000
  Recall: 0.5000

📈 Step 6: Model Comparison Summary
Model Performance Comparison:
Metric          YOLO       ResNet50V2  
-------------------------------------
Accuracy        1.0000     0.7778      
Precision       1.0000     1.0000      
Recall          1.0000     0.5000      
F1 Score        1.0000     0.6667      

✅ Pipeline completed successfully!

🎯 RECOMMENDATION FOR YOUR USE CASE:
Since you need to detect multiple objects in real-world images,
YOLOv8 is the better choice as it can:
  • Detect multiple objects simultaneously
  • Provide bounding box locations
  • Handle varying numbers of objects per image
  • Scale better to production environments
